# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
#: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

### VectorDB Instance

In [4]:
# : Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")



### Collection

In [5]:
# : Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [6]:
# : Create a collection
# Choose any name you want
collection = chroma_client.get_collection(
   name="udaplay-3",
   embedding_function=embedding_fn
)

### Add documents

In [7]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [9]:
collection.count()

15

In [18]:
queries = [
    'Only games that are released in the year 2015?',
    'List of games as racing',
    'Any games from Nintendo'
]


for q in queries:

    results = collection.query(
        query_texts=q,
        include=['metadatas', 'documents', 'distances'],
        n_results=3,
    )

    metadatas = results.get('metadatas', [[]])[0] 

    print("==============Query:", q, "=====================")

    length = 1

    for metadata in metadatas:
        print(length, "\n")
        length = length+1
        print("Platform: ", metadata.get("Platform"))
        print("Name: ", metadata.get("Name"))
        print("YearOfRelease: ", metadata.get("YearOfRelease"))
        print("Description: ", metadata.get("Description"))
        print("Genre: ", metadata.get("Genre"))
        print("Publisher: ", metadata.get("Publisher"))
        print("\n")



==============Query: Only games that are released in the year 2015? =====================
1 

Platform:  PlayStation 3
Name:  Gran Turismo 5
YearOfRelease:  2010
Description:  A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
Genre:  Racing
Publisher:  Sony Computer Entertainment


2 

Platform:  Xbox One
Name:  Minecraft
YearOfRelease:  2014
Description:  A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.
Genre:  Sandbox, Survival
Publisher:  Mojang Studios


3 

Platform:  Nintendo Switch
Name:  Mario Kart 8 Deluxe
YearOfRelease:  2017
Description:  An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.
Genre:  Racing
Publisher:  Nintendo


==============Query: List of games as racing =====================
1 

Platform:  PlayStation 3
Name:  Gran Turismo 5
YearOfRelease:  2010
Description:  A comprehensive racing 